In [ ]:
import pandas as pd
import jenkspy
from pathlib import Path
import numpy as np

# ============================================================
# INPUT
# ============================================================
INPUT_CSV = Path("~/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/RiskScoreModel/archive/analysis_scripts/district_seasonal_risk_score_2025_apr_jul.csv")

df = pd.read_csv(INPUT_CSV)

# ============================================================
# FILTER SUMMER 2026
# ============================================================

# summer = df[df["timeperiod"].isin([
#     "2025_04",
#     "2025_05",
#     "2025_06",
#     "2025_07",
# ])].copy()

summer = df[df["timeperiod"].isin([
    "2025_Apr-Jul"
])].copy()

# ============================================================
# DISTRICT-WISE HEAT RISK
# (Change mean() to max() or sum() if desired)
# ============================================================

district_heat = (
    summer.groupby("district", as_index=False)["heat-days-score"]
    .mean()
)

# ============================================================
# NATURAL JENKS BREAKS
# # ============================================================

# N_CLASSES = 5

# breaks = jenkspy.jenks_breaks(
#     district_heat["heat-days-score"],
#     n_classes=N_CLASSES
# )

# district_heat["heat_risk_class"] = pd.cut(
#     district_heat["heat-days-score"],
#     bins=breaks,
#     labels=range(1, N_CLASSES + 1),
#     include_lowest=True
# )

# print(breaks)
# print(district_heat.head())

# ============================================================
# Z-SCORE BINNING
# ============================================================

mean = district_heat["heat-days-score"].mean()
std = district_heat["heat-days-score"].std()

district_heat["z_score"] = (
    district_heat["heat-days-score"] - mean
) / std

district_heat["heat_risk_class"] = pd.cut(
    district_heat["z_score"],
    bins=[-np.inf, -1, -0.5, 0.5, 1, np.inf],
    labels=[1, 2, 3, 4, 5],
    include_lowest=True
).astype(int)

print(f"Mean = {mean:.4f}")
print(f"Std Dev = {std:.4f}")

print(district_heat.head())


# ============================================================
# SAVE
# ============================================================
OUTPUT_CSV = Path("~/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/RiskScoreModel/archive/analysis_scripts/2025_summer_district_hazard_risk_classes.csv")
district_heat.to_csv(OUTPUT_CSV, index=False)

print("Classification completed")
# print(breaks)

print(f"\nSaved {len(district_heat)} districts to:")
print(OUTPUT_CSV.resolve())

Mean = 8.7631
Std Dev = 1.2143
    district  heat-days-score   z_score  heat_risk_class
0     Anugul           8.3650 -0.327836                3
1   Balangir          10.8450  1.714534                5
2  Baleshwar           6.1475 -2.154028                1
3    Bargarh           9.4500  0.565701                4
4    Bhadrak           7.0050 -1.447845                1
Classification completed

Saved 30 districts to:
/home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/RiskScoreModel/archive/analysis_scripts/~/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/RiskScoreModel/archive/analysis_scripts/2025_summer_district_hazard_risk_classes.csv


In [ ]:
# monthwise heat days score to compare with imd atlas

In [2]:
import pandas as pd
from pathlib import Path

# ============================================================
# INPUT
# ============================================================

INPUT_CSV = Path(
    "~/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/RiskScoreModel/data/district_final_risk_score.csv"
).expanduser()

df = pd.read_csv(INPUT_CSV)

# Extract month
df["month"] = df["timeperiod"].str[-2:]

# ============================================================
# APRIL
# ============================================================

april = (
    df[df["month"] == "04"]
    .groupby("district")["heat-days-score"]
    .sum()
)

# ============================================================
# MAY
# ============================================================

may = (
    df[df["month"] == "05"]
    .groupby("district")["heat-days-score"]
    .sum()
)

# ============================================================
# JUNE
# ============================================================

june = (
    df[df["month"] == "06"]
    .groupby("district")["heat-days-score"]
    .sum()
)

# ============================================================
# ALL MONTHS (JAN-DEC)
# ============================================================

all_months = (
    df[df["month"].isin([f"{i:02d}" for i in range(1, 13)])]
    .groupby("district")["heat-days-score"]
    .sum()
)

# ============================================================
# COMBINE
# ============================================================

district_heat = pd.concat(
    [april, may, june, all_months],
    axis=1
).reset_index()

district_heat.columns = [
    "district",
    "April",
    "May",
    "June",
    "All_Months_Sum"
]

# ============================================================
# SAVE
# ============================================================

OUTPUT_CSV = Path(
    "~/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/RiskScoreModel/archive/analysis_scripts/district_heat_days_summary.csv"
).expanduser()

district_heat.to_csv(OUTPUT_CSV, index=False)

print(district_heat.head())
print(f"\nSaved {len(district_heat)} districts to:")
print(OUTPUT_CSV.resolve())

    district  April    May   June  All_Months_Sum
0     Anugul  93.48  61.66  89.85         1014.51
1   Balangir  75.28  59.70  67.27          949.72
2  Baleshwar  90.22  45.25  97.31          920.02
3    Bargarh  70.97  65.97  66.15          923.79
4    Bhadrak  95.91  43.46  92.52          942.06

Saved 30 districts to:
/home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/RiskScoreModel/archive/analysis_scripts/district_heat_days_summary.csv


In [5]:
"""
Single seasonal heat-risk score per district, for April-July 202x.

Reads district_final_risk_score.csv (one row per district x month), filters
to the 2025_04 .. 2025_07 season, collapses each district's four monthly rows
into ONE season-level row, and then computes ONE score per district for:

    1. exposure            (season-mean population, z-scored across districts)
    2. government-response (total season tender spend, z-scored, inverted)
    3. vulnerability        (DEA efficiency across districts -> Jenks)
    4. heat-hazard          (season-mean heat-days score, z-scored)
    5. topsis-score / heat-risk-score (weighted TOPSIS across the four scores)

Because there's only one "period" now (the season), the z-scores/DEA/TOPSIS
are computed once across all districts, not per month.

Output: data/district_seasonal_risk_score_2025_apr_jul.csv  (one row per district)
"""

import numpy as np
import pandas as pd
import jenkspy

from pathlib import Path
from sklearn.preprocessing import MinMaxScaler

from pulp import (
    LpProblem,
    LpVariable,
    LpMaximize,
    lpSum,
    PULP_CBC_CMD,
    value,
)

# =============================================================================
# INPUT / OUTPUT
# =============================================================================

INPUT_CSV = Path("~/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/RiskScoreModel/data/district_final_risk_score.csv") 
OUTPUT_CSV = Path("~/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/RiskScoreModel/archive/analysis_scripts/district_seasonal_risk_score_2026_apr_jul.csv")

TIMEPERIODS = ["2026_04", "2026_05", "2026_06"]
SEASON_LABEL = "2026_Apr-Jul"

WEIGHTS = {
    "heat-hazard": 4,
    "exposure": 1,
    "vulnerability": 2,
    "government-response": 2,
}
TOTAL_WEIGHT = sum(WEIGHTS.values())

# =============================================================================
# LOAD + FILTER TO SEASON
# =============================================================================

df = pd.read_csv(INPUT_CSV)

df["district"] = df["district"].astype(str).str.strip()
df["timeperiod"] = df["timeperiod"].astype(str).str.strip()

df = df[df["timeperiod"].isin(TIMEPERIODS)].copy()

print("Filtered shape (2025_04 - 2025_07, all months):", df.shape)

if df.empty:
    raise ValueError(
        "No rows found for 2025_04 - 2025_07. "
        "Check that 'timeperiod' values use the 'YYYY_MM' format."
    )

missing_months = set(TIMEPERIODS) - set(df["timeperiod"].unique())
if missing_months:
    print(f"WARNING: missing months in input data: {sorted(missing_months)}")

# =============================================================================
# COLLAPSE 4 MONTHLY ROWS -> 1 SEASONAL ROW PER DISTRICT
# =============================================================================
# - population / rates / counts that describe a *state* (not a flow):
#       averaged over the season
# - tender value (a *flow*, money spent per month):
#       summed over the season -> total season spend
# - heat-days-score: averaged over the season (typical heat burden of the season)

agg_dict = {
    "sum-population": "mean",
    "total-tender-awarded-value": "sum",
    "sum-aged-population": "mean",
    "sum-young-population": "mean",
    "nosanitation-hhds-pct": "mean",
    "workers-affected-pct": "mean",
    "pct-ncd": "mean",
    "health-centres-count": "mean",
    "avg-electricity": "mean",
    "piped-hhds-pct": "mean",
    "heat-days-score": "mean",
}

if "dtname" in df.columns:
    agg_dict["dtname"] = "first"
if "object-id" in df.columns:
    agg_dict["object-id"] = "first"

season_df = df.groupby("district", as_index=False).agg(agg_dict)
season_df["timeperiod"] = SEASON_LABEL

print("\nSeason-level rows (one per district):", len(season_df))

# =============================================================================
# SHARED HELPERS
# =============================================================================

def zscore(x):
    std = x.std(ddof=0)
    if std == 0:
        return pd.Series(0, index=x.index)
    return (x - x.mean()) / std


def classify_standard(z):
    if z <= -1.5:
        return 1
    elif z <= -0.5:
        return 2
    elif z <= 0.5:
        return 3
    elif z <= 1.5:
        return 4
    else:
        return 5


def classify_inverted(z):
    if z <= -1.5:
        return 5
    elif z <= -0.5:
        return 4
    elif z <= 0.5:
        return 3
    elif z <= 1.5:
        return 2
    else:
        return 1


def classify_topsis(score):
    if score <= 0.2:
        return 1
    elif score <= 0.4:
        return 2
    elif score <= 0.6:
        return 3
    elif score <= 0.8:
        return 4
    else:
        return 5


def dea_crs(dmu_df, input_cols, output_cols):
    dmus = dmu_df.index.tolist()
    efficiencies = []

    for dmu in dmus:
        prob = LpProblem(f"DEA_{dmu}", LpMaximize)

        u = {col: LpVariable(f"u_{col}_{dmu}", lowBound=1e-6) for col in output_cols}
        v = {col: LpVariable(f"v_{col}_{dmu}", lowBound=1e-6) for col in input_cols}

        prob += lpSum(u[r] * dmu_df.loc[dmu, r] for r in output_cols)
        prob += lpSum(v[i] * dmu_df.loc[dmu, i] for i in input_cols) == 1

        for j in dmus:
            prob += (
                lpSum(u[r] * dmu_df.loc[j, r] for r in output_cols)
                - lpSum(v[i] * dmu_df.loc[j, i] for i in input_cols)
                <= 0
            )

        prob.solve(PULP_CBC_CMD(msg=False))

        eff = value(prob.objective)
        if eff is None:
            eff = np.nan
        else:
            eff = max(0.0, min(1.0, float(eff)))

        efficiencies.append(eff)

    return efficiencies


def assign_jenks_with_handling(data, n_classes=5):
    data = pd.Series(data)
    unique_vals = np.unique(data)

    if len(unique_vals) == 1:
        return pd.Series([3] * len(data), index=data.index)

    if len(unique_vals) < n_classes:
        n_classes = len(unique_vals)

    while n_classes >= 2:
        try:
            breaks = jenkspy.jenks_breaks(data.values, n_classes=n_classes)
            unique_breaks = np.unique(breaks)

            if len(unique_breaks) < len(breaks):
                n_classes -= 1
                continue

            classes = pd.cut(
                data,
                bins=unique_breaks,
                labels=list(range(1, n_classes + 1)),
                include_lowest=True,
            )
            return classes.astype(int)

        except Exception:
            n_classes -= 1

    return pd.Series([3] * len(data), index=data.index)


# =============================================================================
# 1. EXPOSURE  (season-mean population, z-scored ONCE across districts)
# =============================================================================

season_df["population-z"] = zscore(season_df["sum-population"])
season_df["exposure"] = season_df["population-z"].apply(classify_standard)

# =============================================================================
# 2. GOVERNMENT RESPONSE  (total season tender spend, z-scored, inverted)
# =============================================================================

season_df["govtresponse-z"] = zscore(season_df["total-tender-awarded-value"])
season_df["government-response"] = season_df["govtresponse-z"].apply(classify_inverted)

# =============================================================================
# 3. VULNERABILITY  (DEA CRS efficiency across districts, ONE run -> Jenks)
# =============================================================================

scale_cols = [
    "sum-aged-population",
    "sum-young-population",
    "nosanitation-hhds-pct",
    "workers-affected-pct",
    "pct-ncd",
    "health-centres-count",
    "avg-electricity",
    "piped-hhds-pct",
]

dea_input_df = season_df.copy()

scaler = MinMaxScaler()
dea_input_df[scale_cols] = scaler.fit_transform(dea_input_df[scale_cols])
dea_input_df[scale_cols] += 1e-6

dea_input_df["inv-health-centres-count"] = 1 - dea_input_df["health-centres-count"]
dea_input_df["inv-avg-electricity"] = 1 - dea_input_df["avg-electricity"]
dea_input_df["inv-piped-hhds-pct"] = 1 - dea_input_df["piped-hhds-pct"]
dea_input_df["constant-output"] = 1.0

INPUTS = [
    "sum-aged-population",
    "sum-young-population",
    "nosanitation-hhds-pct",
    "workers-affected-pct",
    "pct-ncd",
    "inv-health-centres-count",
    "inv-avg-electricity",
    "inv-piped-hhds-pct",
]
OUTPUTS = ["constant-output"]

dea_input_df.index = dea_input_df["district"]

print("\nRunning DEA across districts for the season...")
season_df["efficiency"] = dea_crs(dea_input_df, INPUTS, OUTPUTS)

season_df["vulnerability-raw"] = 1 - season_df["efficiency"]
season_df["vulnerability"] = assign_jenks_with_handling(
    season_df["vulnerability-raw"], n_classes=5
)

# =============================================================================
# 4. HEAT HAZARD  (season-mean heat-days score, z-scored ONCE across districts)
# =============================================================================

season_df["heat-zscore"] = zscore(season_df["heat-days-score"])
season_df["heat-hazard"] = season_df["heat-zscore"].apply(classify_standard)

# =============================================================================
# 5. TOPSIS COMBINATION -> ONE heat-risk-score per district
# =============================================================================

norm = pd.DataFrame(index=season_df.index)
for col in WEIGHTS:
    min_v, max_v = season_df[col].min(), season_df[col].max()
    norm[col] = 0 if max_v == min_v else (season_df[col] - min_v) / (max_v - min_v)

for col in WEIGHTS:
    norm[col] *= WEIGHTS[col] / TOTAL_WEIGHT

ideal_best = norm.max()
ideal_worst = norm.min()

dist_best = np.sqrt(((norm - ideal_best) ** 2).sum(axis=1))
dist_worst = np.sqrt(((norm - ideal_worst) ** 2).sum(axis=1))

season_df["topsis-score"] = dist_worst / (dist_best + dist_worst)
season_df["heat-risk-score"] = season_df["topsis-score"].apply(classify_topsis)

# =============================================================================
# ROUNDING
# =============================================================================

int_cols = [c for c in ["sum-aged-population", "sum-young-population", "sum-population"] if c in season_df.columns]
if int_cols:
    season_df[int_cols] = season_df[int_cols].round().astype("Int64")

numeric_cols = season_df.select_dtypes(include="number").columns
decimal_cols = numeric_cols.difference(int_cols)
season_df[decimal_cols] = season_df[decimal_cols].round(3)

# =============================================================================
# SAVE
# =============================================================================

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
season_df.to_csv(OUTPUT_CSV, index=False)

print(f"\nSaved: {OUTPUT_CSV}")
print(f"Rows: {len(season_df)} (one per district)  Columns: {len(season_df.columns)}")

print("\nExposure distribution:")
print(season_df["exposure"].value_counts().sort_index())

print("\nGovernment response distribution:")
print(season_df["government-response"].value_counts().sort_index())

print("\nVulnerability distribution:")
print(season_df["vulnerability"].value_counts().sort_index())

print("\nHeat hazard distribution:")
print(season_df["heat-hazard"].value_counts().sort_index())

print("\nHeat risk score distribution:")
print(season_df["heat-risk-score"].value_counts().sort_index())

print("\nPreview:")
print(
    season_df[
        [
            "district",
            "timeperiod",
            "heat-hazard",
            "exposure",
            "vulnerability",
            "government-response",
            "topsis-score",
            "heat-risk-score",
        ]
    ].sort_values("heat-risk-score", ascending=False)
)

Filtered shape (2025_04 - 2025_07, all months): (90, 29)

Season-level rows (one per district): 30

Running DEA across districts for the season...

Saved: ~/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/RiskScoreModel/archive/analysis_scripts/district_seasonal_risk_score_2026_apr_jul.csv
Rows: 30 (one per district)  Columns: 26

Exposure distribution:
exposure
2    10
3    11
4     6
5     3
Name: count, dtype: int64

Government response distribution:
government-response
3    30
Name: count, dtype: int64

Vulnerability distribution:
vulnerability
1    24
2     2
3     1
4     1
5     2
Name: count, dtype: int64

Heat hazard distribution:
heat-hazard
2    11
3     7
4     9
5     3
Name: count, dtype: int64

Heat risk score distribution:
heat-risk-score
1    10
2     8
3     8
4     3
5     1
Name: count, dtype: int64

Preview:
          district    timeperiod  heat-hazard  exposure  vulnerability  \
12         Jajapur  2026_Apr-Jul            5         4              5   
8        Dhenk